In [1]:
import duckdb
import pandas as pd
import os

print("Libraries loaded successfully")
print(f"Working directory: {os.getcwd()}")
print(f"DuckDB version: {duckdb.__version__}")

Libraries loaded successfully
Working directory: g:\airbnb-data-engineering-assessment\notebooks
DuckDB version: 1.1.3


In [2]:
import urllib.request
import os

os.makedirs('../data/raw', exist_ok=True)

urls = {
    'listings.csv.gz': 'http://data.insideairbnb.com/united-kingdom/england/london/2024-09-06/data/listings.csv.gz',
    'reviews.csv.gz': 'http://data.insideairbnb.com/united-kingdom/england/london/2024-09-06/data/reviews.csv.gz',
}

for filename, url in urls.items():
    filepath = f'../data/raw/{filename}'
    if not os.path.exists(filepath):
        print(f"Downloading {filename}...")
        urllib.request.urlretrieve(url, filepath)
        size_mb = os.path.getsize(filepath) / 1024 / 1024
        print(f"Done ({size_mb:.1f} MB)")
    else:
        size_mb = os.path.getsize(filepath) / 1024 / 1024
        print(f"{filename} already exists ({size_mb:.1f} MB)")

Done (48.4 MB)
Done (218.9 MB)


In [3]:
# Check downloaded files

import os
for f in os.listdir('../data/raw'):
    size_mb = os.path.getsize(f'../data/raw/{f}') / 1024 / 1024
    print(f"  📄 {f}: {size_mb:.1f} MB")

  📄 listings.csv.gz: 48.4 MB
  📄 reviews.csv.gz: 218.9 MB


In [4]:
# Load CSVs into DuckDB

# Connect to a database file (will be created if doesn't exist)
con = duckdb.connect('../data/airbnb.db')
print("Connected to DuckDB database!")

# Load listings.csv into a DuckDB table
con.execute("""
    CREATE TABLE listings AS 
    SELECT * FROM read_csv_auto('../data/raw/listings.csv.gz')
""")
print("Listings table created!")

# Load reviews.csv into a DuckDB table
con.execute("""
    CREATE TABLE reviews AS 
    SELECT * FROM read_csv_auto('../data/raw/reviews.csv.gz')
""")
print("Reviews table created!")

Connected to DuckDB database!
Listings table created!


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Reviews table created!


Counts for listings and reviews

In [13]:
listings_count = con.execute("SELECT COUNT(*) FROM listings").fetchone()[0]
reviews_count = con.execute("SELECT COUNT(*) FROM reviews").fetchone()[0]
listings_cols = len(con.execute("DESCRIBE listings").fetchall())
reviews_cols = len(con.execute("DESCRIBE reviews").fetchall())

print(f"\n Listings: {listings_count:,} rows, {listings_cols} columns")
print(f" Reviews: {reviews_count:,} rows, {reviews_cols} columns")


 Listings: 96,182 rows, 75 columns
 Reviews: 1,887,519 rows, 6 columns


Schema for listings

In [8]:
con.execute("DESCRIBE listings").df()

,column_name,column_type,null,key,default,extra
0,id,BIGINT,YES,None,None,None
1,listing_url,VARCHAR,YES,None,None,None
2,scrape_id,BIGINT,YES,None,None,None
3,last_scraped,DATE,YES,None,None,None
4,source,VARCHAR,YES,None,None,None
...,...,...,...,...,...,...
70,calculated_host_listings_count,BIGINT,YES,None,None,None
71,calculated_host_listings_count_entire_homes,BIGINT,YES,None,None,None
72,calculated_host_listings_count_private_rooms,BIGINT,YES,None,None,None
73,calculated_host_listings_count_shared_rooms,BIGINT,YES,None,None,None


Schema for reviews

In [9]:
con.execute("DESCRIBE reviews").df()

,column_name,column_type,null,key,default,extra
0,listing_id,BIGINT,YES,None,None,None
1,id,BIGINT,YES,None,None,None
2,date,DATE,YES,None,None,None
3,reviewer_id,BIGINT,YES,None,None,None
4,reviewer_name,VARCHAR,YES,None,None,None
5,comments,VARCHAR,YES,None,None,None


Sample data of listings (First 5)

In [11]:
con.execute("SELECT * FROM listings LIMIT 5").df()

,id,listing_url,scrape_id,last_scraped,source,name,description,neighborhood_overview,picture_url,host_id,...,review_scores_communication,review_scores_location,review_scores_value,license,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month
0,13913,https://www.airbnb.com/rooms/13913,20240906025501,2024-09-06,city scrape,Holiday London DB Room Let-on going,My bright double bedroom with a large window h...,Finsbury Park is a friendly melting pot commun...,https://a0.muscache.com/pictures/miso/Hosting-...,54730,...,4.84,4.72,4.72,None,False,3,2,1,0,0.26
1,15400,https://www.airbnb.com/rooms/15400,20240906025501,2024-09-07,city scrape,Bright Chelsea Apartment. Chelsea!,Lots of windows and light. St Luke's Gardens ...,It is Chelsea.,https://a0.muscache.com/pictures/428392/462d26...,60302,...,4.84,4.93,4.75,None,False,1,1,0,0,0.54
2,17402,https://www.airbnb.com/rooms/17402,20240906025501,2024-09-07,city scrape,Fab 3-Bed/2 Bath & Wifi: Trendy W1,"You'll have a great time in this beautiful, cl...","Fitzrovia is a very desirable trendy, arty and...",https://a0.muscache.com/pictures/39d5309d-fba7...,67564,...,4.72,4.89,4.61,None,False,6,6,0,0,0.34
3,24328,https://www.airbnb.com/rooms/24328,20240906025501,2024-09-07,city scrape,"Battersea live/work artist house, garden & par...","Artist house, bright high ceiling rooms for bo...","- Battersea is a quiet family area, easy acces...",https://a0.muscache.com/pictures/9194b40f-c627...,41759,...,4.93,4.59,4.65,None,False,1,1,0,0,0.56
4,33332,https://www.airbnb.com/rooms/33332,20240906025501,2024-09-06,city scrape,Beautiful Ensuite Richmond-upon-Thames borough,"Walking distance to Twickenham Stadium, 35 min...",Peaceful and friendly.,https://a0.muscache.com/pictures/miso/Hosting-...,144444,...,4.50,4.67,4.22,None,False,2,0,2,0,0.11


Sample data of reviews (First 5)

In [12]:
con.execute("SELECT * FROM reviews LIMIT 5").df()

,listing_id,id,date,reviewer_id,reviewer_name,comments
0,13913,80770,2010-08-18,177109,Michael,My girlfriend and I hadn't known Alina before ...
1,13913,367568,2011-07-11,19835707,Mathias,Alina was a really good host. The flat is clea...
2,13913,529579,2011-09-13,1110304,Kristin,Alina is an amazing host. She made me feel rig...
3,13913,595481,2011-10-03,1216358,Camilla,"Alina's place is so nice, the room is big and ..."
4,13913,612947,2011-10-09,490840,Jorik,"Nice location in Islington area, good for shor..."


#### Mapped relationships from listings and reviews

Primary & Foreign Keys

| Table | Column | Key Type | References |
|-------|--------|----------|------------|
| listings | id | Primary Key | — |
| reviews | id | Primary Key | — |
| reviews | listing_id | Foreign Key | listings.id |


#### Relationship Type
One-to-Many (1:N) — One listing can have many reviews over time, but each review belongs to exactly one listing.

How to Join;

```sql
SELECT l.id, l.name, l.price, r.date, r.comments
FROM listings l
JOIN reviews r ON l.id = r.listing_id


#### Reviewed official data dictionary

**Source**
Inside Airbnb Data Dictionary v4.3 for `listings.csv` (August 2022)  
URL: https://docs.google.com/spreadsheets/d/1iWCNJcSutYqpULSQHlNyGInUvHg2BoUGoNRIGa6Szc4/

**Columns requiring special interpretation**

| Column | Special Interpretation (from official dictionary) |
|--------|---------------------------------------------------|
| `price` | "$ sign is a technical artifact of the export, please ignore it" — must strip and cast to numeric |
| `minimum_nights` | "calendar rules may be different" — may not reflect actual calendar restrictions |
| `reviews_per_month` | Calculated field — formula: `IF (scrape_date - first_review <= 30) THEN number_of_reviews ELSE number_of_reviews / ((scrape_date - first_review + 1) / (365/12))` |
| `calculated_host_listings_count` | "The number of listings the host has in the current scrape, in the city/region geography" — **city-specific, not global** |
| `availability_365` | "The availability of the listing x days in the future" — future only, not historical |
| `last_review` | Date of MOST RECENT review only |
| `neighbourhood_group_cleansed` | NULL for London — NYC borough structure (Manhattan, Brooklyn) |
| `license` | Optional regulatory field — NULL is valid for London (UK doesn't require display) |
| `room_type` | Only 4 valid values: Entire home/apt, Private room, Shared room, Hotel room |
| `bathrooms_text` | Newer field replacing numeric `bathrooms` — older scrapes used numeric, newer use text |
| `source` | "neighbourhood search" or "previous scrape" — affects whether listing is currently active |
| `host_is_superhost` | Boolean (t/f) — null/missing for hosts not yet evaluated |

#### Limitations

Coverage gaps

- **Single snapshot** — Data from September 6-7, 2024 only. No historical tracking of how listings change over time.
- **No booking data** — Only see availability, not actual confirmed bookings.
- **No revenue data** — Prices shown are listed prices, not actual transaction prices.
- **No calendar file** — I did not download `calendar.csv.gz` (would have enabled hypothesis H5: weekend vs weekday pricing).
- **No neighbourhoods.geojson** — I did not download geographic boundaries (limits geographic visualization options).

Scraping artifacts

- **Host churn** — Listings may appear or disappear between scrapes (Inside Airbnb re-scrapes every ~65 days).
- **Source field variation** — `source` can be "previous scrape" (listing may no longer be active on Airbnb but was confirmed available within last 65 days).
- **Test/internal listings** — Occasionally appear in scrape data.
- **Pseudonymized hosts** — Host names often not real names; privacy-protected.
- **Inconsistent text formatting** — `name` and `description` fields contain emojis, special characters, multiple languages.

Missing historical data

- **No first_review date** in the fields I use commonly eventhough it's in full listings.csv, our queries focus on `last_review`.
- **No price history** — Cannot see how prices changed over time.
- **No availability history** — Only future 365-day availability, not past.
- **No host tenure easily** — `host_since` is available but we focus on `calculated_host_listings_count`.

#### Assumptions

About the data

1. **Currency** — Prices assumed to be in GBP (London listings).
2. **Coordinates valid** if within UK bounding box (~49-61°N, -8 to 2°E).
3. **Price > 0** for price analysis — exclude $0 listings as likely data errors.
4. **Reviews ≈ Bookings proxy** — More reviews ≈ more bookings (rough correlation).
5. **Availability 365 = bookable days** — Not the same as actual booked days.
6. **License column NULL** is valid for London (not legally required in UK).
7. **neighbourhood_group NULL** is valid for London (NYC-specific field).
8. **`source = "previous scrape"`** listings are still considered available.

About the business context

1. **Active listings** — A listing with 0 reviews is treated as "new" not "inactive."
2. **Host activity** — `calculated_host_listings_count` reflects London listings only (per dictionary).
3. **Review text quality** — Some reviews may be empty, very short, or in non-English languages.
4. **Room type standardization** — Assumed `room_type` follows the 4-value taxonomy from the dictionary.

#### Business domain context

Entities

**Listing** — A single property available for short-term rental on Airbnb.  
- Has unique `id`, belongs to one host, located in one neighbourhood  
- Has one room_type, one price, one availability calendar  
- Can accumulate many reviews over time

**Host** — The owner or manager of one or more listings.  
- Has `host_id`, `host_name`, optional `host_since` date  
- May operate multiple listings (`calculated_host_listings_count`)  
- May have `host_is_superhost` status

**Review** — Guest feedback posted after a stay.  
- Has unique `id`, tied to exactly one listing via `listing_id`  
- Has `date`, `reviewer_id`, and `comments` text  
- Contributes to listing's overall review scores

Relationships

**HOST (1) ──────< (N) LISTINGS (1) ──────< (N) REVIEWS via host_id via listing_id**

Business questions this data can answer

- What is the average price per neighbourhood?
- How does superhost status affect review scores?
- Are certain room types more expensive?
- Which neighbourhoods have the most listings?
- How does availability vary across the city?
- What factors predict listing price?
- What are common themes in guest reviews (sentiment)?
- How concentrated is the market (do a few hosts control most listings)?


In [14]:
con.close()
print("Connection closed!")

Connection closed!
